- impute gap < 2
- generate target 
- log transform target
- approche I: generate lag and rolling features
- drop na
train/val split



In [28]:
# Core libraries
import pandas as pd
import numpy as np
import json

#preprocessing
from sklearn import set_config
set_config(display = 'diagram', transform_output= "pandas")

# Sklearn preprocessing
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Model selection
from sklearn.metrics import root_mean_squared_log_error
from sklearn.model_selection import cross_val_score,cross_validate, TimeSeriesSplit
import optuna, optuna_dashboard

#feature selection
from sklearn.inspection import permutation_importance

# pipelines
from sklearn.compose import make_column_transformer, ColumnTransformer
from sklearn.pipeline import make_pipeline, FunctionTransformer

# models
from sklearn.linear_model import ElasticNet #always like to have a linear model for comparison
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical analysis
from scipy import stats


# Project config
from src.params import *
from src.utils import *

# Plot configuration
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
import warnings
warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 0. Load Data
- ingest data from API or from local sources if already downloaded **for the list of cities selected during EDA**

In [46]:
from src.ingestion.openaq import OpenAQClient
from src.ingestion.openweather import OpenWeatherClient

aq_client = OpenAQClient(api_key= API_AQ, radius= 7000)
df_airqual= aq_client.get_data(CITIES,
                   start_date= START_TRAIN_DATE_STR,
                   end_date= END_TRAIN_DATE_STR,
                   start_project_date= START_PROJECT_DATE_STR,
                   end_project_date= END_PROJECT_DATE_STR
                   )

weather_client = OpenWeatherClient(api_key= API_OW)
df_weather = weather_client.get_all_data(cities= CITIES,
                                      start_date= START_TRAIN_DATE_STR,
                                      end_date= END_TRAIN_DATE_STR)


Processing Paris...
  Monitor-grade locations: 8
  Active at end date: 5
  Full coverage: 3
  PM2.5 sensors found: 3
✓ 3 sensor(s) selected for Paris
  Fetching data for 3 sensors...
  [1/3] Sensor 1582598
  └─ Sensor 1582598: using cached data
  [2/3] Sensor 9548
  └─ Sensor 9548: using cached data
  [3/3] Sensor 1562368
  └─ Sensor 1562368: using cached data
  ✓ 2046 measurements extracted

Processing Lyon...
  Monitor-grade locations: 3
  Active at end date: 3
  Full coverage: 3
  PM2.5 sensors found: 3
✓ 3 sensor(s) selected for Lyon
  Fetching data for 3 sensors...
  [1/3] Sensor 8559
  └─ Sensor 8559: using cached data
  [2/3] Sensor 7922
  └─ Sensor 7922: using cached data
  [3/3] Sensor 24867
  └─ Sensor 24867: using cached data
  ✓ 2099 measurements extracted

Processing New York...
  Monitor-grade locations: 6
  Active at end date: 6
  Full coverage: 6
  PM2.5 sensors found: 6
✓ 6 sensor(s) selected for New York
  Fetching data for 6 sensors...
  [1/6] Sensor 673
  └─ Sensor

Fetching Paris weather data:   0%|          | 0/731 [00:00<?, ?it/s]

✅ [Paris] All days fetched! Cached: 731, New: 0

Processing Lyon...


Fetching Lyon weather data:   0%|          | 0/731 [00:00<?, ?it/s]

✅ [Lyon] All days fetched! Cached: 731, New: 0

Processing New York...


...

✅ [London] All days fetched! Cached: 731, New: 0

Processing Berlin...


Fetching Berlin weather data:   0%|          | 0/731 [00:00<?, ?it/s]

✅ [Berlin] All days fetched! Cached: 731, New: 0

Processing Rome...


Fetching Rome weather data:   0%|          | 0/731 [00:00<?, ?it/s]

✅ [Rome] All days fetched! Cached: 731, New: 0
✅ Saved 4386 rows to ../data/raw/weather.csv


In [ ]:
#load airqual
df_airqual = load_data_local(filepath= "../data/raw/airqual_eda.csv", source="airqual")
df_airqual = clean_neg_values(df_airqual)

✅ Loaded 21342 rows from ../data/raw/aq_data.csv
⚠️  359 aberrant (negative or zero) values found (1.68%)
✅ All negative values removed — 20983 rows remaining


In [ ]:


df_weather = load_data_local(filepath= "../data/raw/weather_eda.csv", source= "weather")
data = merge_source_df(df_airqual= df_airqual, df_weather= df_weather)

✅ Loaded 7310 rows from ../data/raw/weather.csv
✅ DataFrames merged successfully — 20983/21470 days have airqual data (97.7%)


In [32]:
#filter only relevant cities selected during EDA
cities = list(CITIES.keys())
print(sorted(cities))
data = data[data["city"].isin(cities)]
print(sorted(data["city"].unique()))


['Berlin', 'London', 'Lyon', 'New York', 'Paris', 'Rome']
['Berlin', 'London', 'Lyon', 'New York', 'Paris', 'Rome']


# I. Impute NaN


In [37]:
data.isna().sum()

city                0
sensor_id         106
date                0
pm25_avg          106
pm25_min          106
pm25_q25          106
pm25_median       106
pm25_q75          106
pm25_max          106
coverage          106
temp_min            0
temp_max            0
temp_avg            0
cloud_cover         0
humidity            0
precipitation       0
pressure            0
wind_speed          0
wind_direction      0
dtype: int64

In [ ]:
neg_imputer = SimpleImputer()

# I. Inpute single space gaps